# IMC analysis of the chordoma TMA (CH)

Raw imaging mass cytometry data of the CH tissue microarray (15 patients, 11 cores each) are processed into single-cell tables:

1. Panel
2. Image stacks from the `.mcd` acquisitions
3. Cell segmentation (Cellpose)
4. Single-cell measurements (Steinbock)
5. Export

Cores are named by the TMA key: `ROI<acquisition number>_<patient><core>`, e.g. `ROI013_E6` (patient E, core 6). Patients A–O are the slide columns 15–1, cores 1–11 the slide rows A–K (1–3 high grade, 4 low grade, 5–6 high infiltration, 7–8 tumour periphery, 9–10 tumour border, 11 non-neoplastic). The key per ROI is in `data/roi_key_map.csv`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

WORKFLOW_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = WORKFLOW_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw' / 'Tiffs'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

# Original acquisition file (imaging facility share, read-only)
MCD_PATH = Path('/Volumes/ricerca/Unita_Imaging/Shared/Hyperion/Projects/Cananzi/20260116_TMA/IMC_data/CH/CH.mcd')

def run(cmd):
    result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{Path(cmd[1]).name} failed')

## 1. Panel

`data/panel.csv` is built from the channel files in `data/raw/Tiffs/`. Segmentation channels: nuclear DNA1, DNA2, Brachyury; membrane S100, CD68, aSMA, CD31, CD3, CD66b. Technical channels (127I, 131Xe, 134Xe, 138Ba, 208Pb, 80ArAr) are excluded from analysis.

In [ ]:
run([
    sys.executable, str(SCRIPTS_DIR / '01_create_panel_from_raw.py'),
    '--raw-dir', str(RAW_DIR),
    '--output', str(DATA_DIR / 'panel.csv'),
    '--numbered-copy', str(WORKFLOW_DIR / 'results' / '01_panel.csv'),
    '--availability-output', str(DATA_DIR / 'roi_marker_availability.csv'),
    '--numbered-availability-copy', str(WORKFLOW_DIR / 'results' / '01_roi_marker_availability.csv'),
    '--nuclear-marker', 'DNA1', 'DNA2', 'Brachiury',
    '--membrane-marker', 'S100', 'CD68', 'aSMA', 'CD31', 'CD3', 'CD66b',
    '--technical-channel', '127I', '131Xe', '134Xe', '138Ba', '208Pb', '80ArAr',
])

## 2. Image stacks

Pixel values are read from `CH.mcd` as float32; the facility TIFF export stores counts truncated to integers. Each ROI is matched to the acquisition with the same number. Hot pixels (more than 50 counts above all 8 neighbours) are replaced by the neighbourhood maximum. Images are named by the TMA key (`data/roi_key_map.csv`).

In [ ]:
run([
    sys.executable, str(SCRIPTS_DIR / '02_preprocess_imc_images.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--raw-dir', str(RAW_DIR),
    '--mcd', str(MCD_PATH),
    '--hpf', '50',
    '--skip-existing',
    '--numbered-images-copy', 'results/02_images.csv',
    '--numbered-inventory', 'results/03_preprocessed_tiff_inventory.csv',
    '--log-file', 'logs/02_preprocess_imc_images.log',
])

## 3. Cell segmentation

Cellpose (`cpsam_v2`) on a nuclear and a membrane composite (pixel-wise maximum of the segmentation channels, percentile-normalised per image). Flow threshold 0.4, cell probability threshold 0, minimum size 20 px (1 px = 1 µm).

In [ ]:
run([
    sys.executable, str(SCRIPTS_DIR / '03_segment_cellpose.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--model', 'cpsam_v2',
    '--flow-threshold', '0.4',
    '--cellprob-threshold', '0.0',
    '--min-size', '20',
    '--normalization', 'percentile',
    '--nuclear-aggregation', 'max',
    '--membrane-aggregation', 'max',
    '--gpu',
    '--numbered-inventory', 'results/04_cellpose_mask_inventory.csv',
    '--log-file', 'logs/03_segment_cellpose.log',
])

## 4. Single-cell measurements

Steinbock (Docker): mean intensity of each channel per cell, cell morphology (area, centroid, axis lengths, eccentricity), and neighbours (cells whose borders, expanded by 4 px, touch).

In [ ]:
for script, inventory, log, extra in [
    ('04_measure_intensities.py', '05_intensity_table_inventory.csv', '04_measure_intensities.log', []),
    ('05_measure_regionprops.py', '06_regionprops_table_inventory.csv', '05_measure_regionprops.log', []),
    ('06_measure_neighbors.py', '07_neighbors_table_inventory.csv', '06_measure_neighbors.log',
     ['--neighbor-type', 'expansion', '--dmax', '4']),
]:
    run([
        sys.executable, str(SCRIPTS_DIR / script),
        '--workflow-dir', str(WORKFLOW_DIR),
        *extra,
        '--numbered-inventory', f'results/{inventory}',
        '--log-file', f'logs/{log}',
    ])

## 5. Export

One row per cell in `data/cells.csv` and `data/cells.h5ad` (copies `results/08_cells.csv`, `results/09_cells.h5ad`), with the TMA key added: patient, core and sample type.

In [ ]:
run([sys.executable, str(SCRIPTS_DIR / '07_export_data.py'), '--workflow-dir', str(WORKFLOW_DIR)])

In [ ]:
cells = pd.read_csv(DATA_DIR / 'cells.csv', usecols=['Image', 'patient_id', 'sample_type'])
print('Images:', cells['Image'].nunique(), '| Cells:', len(cells))
print(cells.groupby('sample_type').size())